In [ ]:
import torch
from torch import nn

In [80]:
from torch.utils.data import DataLoader
import torchaudio

data = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./")

data_train = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./", subset="training")
data_validation = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./", subset="validation")
data_testing = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./", subset="testing")

def pre_process_batch(batch):
    target_len = 16000
    new_batch = []
    
    for waveform, sample_rate, label_str, speaker_id, utterance_number in batch:

        #Ici on rajoute du bruit blanc en entree
        current_len = waveform.shape[1]
        if current_len < target_len:
            waveform = torch.nn.functional.pad(waveform, (0, target_len - current_len))
        
        #On reconstruit l'échantillon, avec l'indice du labels
        new_batch.append((waveform, labels.index(label_str)))

    waveforms = torch.stack([item[0] for item in new_batch])
    labels_tensor = torch.tensor([item[1] for item in new_batch])
    
    return waveforms, labels_tensor
 
train_loader = DataLoader(
    data_train, 
    batch_size=32, 
    shuffle=True,
    collate_fn=pre_process_batch 
)

val_loader = DataLoader(
    data_validation, 
    batch_size=32, 
    shuffle=True,
    collate_fn=pre_process_batch 
)

#for batch_idx, (waveforms, labels_batch) in enumerate(train_loader):
#    print(batch_idx, waveforms, labels_batch)
#    break

In [81]:
import os
labels = os.listdir("./SpeechCommands/speech_commands_v0.02/")

labels.remove("README.md")
labels.remove("LICENSE")
labels.remove("testing_list.txt")
labels.remove("validation_list.txt")
labels.remove("_background_noise_")
labels.remove(".DS_Store")

print("Il y a  : ", len(labels), " labels")
print(labels)

Il y a  :  35  labels
['right', 'eight', 'tree', 'go', 'wow', 'happy', 'cat', 'dog', 'forward', 'bed', 'backward', 'zero', 'off', 'yes', 'learn', 'five', 'seven', 'sheila', 'left', 'nine', 'on', 'bird', 'no', 'follow', 'six', 'one', 'down', 'house', 'two', 'four', 'up', 'visual', 'stop', 'three', 'marvin']


In [77]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(1, 16, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv1d(16, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv1d(32, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv1d(64, 128, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 1000, 1)
        )

    def forward(self, x):
        return self.net(x)

In [88]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, n_words=35):
        super().__init__()

        self.label_emb = nn.Embedding(n_words, 32)

        self.fc = nn.Linear(latent_dim + 32, 128 * 125)

        self.net = nn.Sequential(
            nn.ConvTranspose1d(128, 64, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose1d(64, 32, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose1d(32, 16, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose1d(16, 8, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose1d(8, 4, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose1d(4, 2, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose1d(2, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z, labels):
        # z: (B, 100)
        # labels: (B,)

        c = self.label_emb(labels)      # (B, 32)
        x = torch.cat([z, c], dim=1)    # (B, 132)

        x = self.fc(x)                 # (B, 128*125)
        x = x.view(-1, 128, 125)       # (B, 128, 125)

        x = self.net(x)                # (B, 1, 16000)
        return x

In [89]:
import torch
import torch.nn as nn

device = "cpu"

G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCEWithLogitsLoss()

opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

latent_dim = 100

In [91]:
num_epochs = 20

print("labels_batch min/max:", labels_batch.min().item(), labels_batch.max().item())
print("embedding size:", G.label_emb.num_embeddings)

for epoch in range(num_epochs):

    for i, (waveforms, labels_batch) in enumerate(train_loader):

        waveforms = waveforms.to(device)          # (B, 1, 16000)
        labels_batch = labels_batch.to(device)

        B = waveforms.size(0)

        # =========================
        # 1. Train Discriminator
        # =========================
        z = torch.randn(B, latent_dim).to(device)

        fake = G(z, labels_batch)

        real_logits = D(waveforms)
        fake_logits = D(fake.detach())

        real_targets = torch.ones_like(real_logits)
        fake_targets = torch.zeros_like(fake_logits)

        loss_D = criterion(real_logits, real_targets) + \
                 criterion(fake_logits, fake_targets)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # =========================
        # 2. Train Generator
        # =========================
        z = torch.randn(B, latent_dim).to(device)

        fake = G(z, labels_batch)
        fake_logits = D(fake)

        loss_G = criterion(fake_logits, torch.ones_like(fake_logits))

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        # =========================
        # LOGS
        # =========================
        if i % 50 == 0:
            print(f"[{epoch}/{num_epochs}] step {i} "
                  f"loss_D={loss_D.item():.4f} loss_G={loss_G.item():.4f}")

        if i % 200 == 0:
            torchaudio.save(
                f"fake_{epoch}_{i}.wav",
                fake[0].detach().cpu(),
                16000
            )

labels_batch min/max: 1 30
embedding size: 35
[0/20] step 0 loss_D=0.0500 loss_G=4.1823
[0/20] step 50 loss_D=0.0009 loss_G=7.8604
[0/20] step 100 loss_D=0.0064 loss_G=11.3130
[0/20] step 150 loss_D=0.0007 loss_G=11.6021
[0/20] step 200 loss_D=0.0006 loss_G=8.5287
[0/20] step 250 loss_D=0.0006 loss_G=13.2381
[0/20] step 300 loss_D=0.0000 loss_G=11.7008
[0/20] step 350 loss_D=0.0004 loss_G=8.2868
[0/20] step 400 loss_D=0.0001 loss_G=9.0648
[0/20] step 450 loss_D=0.0004 loss_G=8.8004
[0/20] step 500 loss_D=0.0014 loss_G=7.3282
[0/20] step 550 loss_D=0.0006 loss_G=8.5001
[0/20] step 600 loss_D=0.0006 loss_G=8.6789
[0/20] step 650 loss_D=0.0003 loss_G=9.4604
[0/20] step 700 loss_D=0.0005 loss_G=8.7009
[0/20] step 750 loss_D=0.0348 loss_G=7.4093
[0/20] step 800 loss_D=0.0025 loss_G=6.0072
[0/20] step 850 loss_D=0.0002 loss_G=11.2079
[0/20] step 900 loss_D=0.0011 loss_G=11.9747
[0/20] step 950 loss_D=0.0012 loss_G=11.1566
[0/20] step 1000 loss_D=0.0006 loss_G=10.3080
[0/20] step 1050 loss_D=

KeyboardInterrupt: 